In [ ]:
#WEIGHTED BY VCUT
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import r2_score, mean_squared_error


# Set random seeds for reproducibility
torch.manual_seed(0)
np.random.seed(1)

# Load data
##### CONVERT TXT TO DATA ARRAY #######
data = np.genfromtxt("../data/vcut_results_merged_OLD.txt", skip_header=1,delimiter=",")
data = data[data[:, 4] != 0]
print(np.shape(data))

# Inputs: first 3 columns;
##### TARGET ~20 RIGHTMOST COLUMNS ######
X = data[:, :3]
Y = data[:, 3:]

# Convert to tensors with float32 dtype
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

# compute from TRAIN set only
X_mu, X_sigma = X_tensor.mean(0), X_tensor.std(0)
Y_mu, Y_sigma = Y_tensor.mean(0), Y_tensor.std(0)

def normX(X): return (X - X_mu)/X_sigma
def normy(Y): return (Y - Y_mu)/Y_sigma
def denormy(Yhat): return Yhat*Y_sigma + Y_mu

# Train/test split (80/20)
n_samples = len(X_tensor)
n_train = int(0.8 * n_samples)
n_test = n_samples - n_train
dataset = TensorDataset(X_tensor, Y_tensor)
train_set, test_set = random_split(dataset, [n_train, n_test])

train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=len(test_set))

# Neural Network Definition
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim=3, output_dim=1, width=32, depth=3, activation='selu'):
        super().__init__()
        if activation == 'relu':
            act = nn.ReLU()
        elif activation == 'tanh':
            act = nn.Tanh()
        elif activation == 'silu':
          act = nn.SiLU()
        elif activation == 'sigmoid':
          act = nn.Sigmoid()
        elif activation == 'selu':
          act = nn.SELU()
        elif activation == 'elu':
          act = nn.ELU()
        elif activation == 'softplus':
          act = nn.Softplus()
        else:
            raise ValueError("activation not included")

        layers = [nn.Linear(input_dim, width), act]
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(act)
        layers.append(nn.Linear(width, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Instantiate model, loss, and optimizer
model = NeuralNetwork(input_dim=3, output_dim=20, width=75, depth=3, activation='silu')
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Training Loop
n_epochs = 500
eps = 1e-10

for epoch in range(n_epochs):
    model.train()
    total_loss = 0.0

    for X_batch, Y_batch in train_loader:
        optimizer.zero_grad()

        # Forward pass
        Y_pred = model(normX(X_batch))

        # relative MSE
        Y_pred_denorm = denormy(Y_pred)
        loss = torch.mean(((Y_pred_denorm - Y_batch) / (Y_batch + eps)) ** 2)


        # Backprop and update
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Logging
    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d}, Loss = {total_loss/len(train_loader):.6f}")

# Evaluation
model.eval()
with torch.no_grad():
    for X_test, Y_test in test_loader:
        Y_pred = model(normX(X_test))
        final_loss = criterion(Y_pred, normy(Y_test)).item()


# Convert to numpy for metrics
Y_true = Y_test.detach().numpy()
Y_pred_np = denormy(Y_pred).detach().numpy()
r2 = r2_score(Y_true, Y_pred_np)
rmse = np.sqrt(mean_squared_error(Y_true, Y_pred_np))
rel_rmse = np.sqrt(np.mean(((Y_pred_np - Y_true) / (Y_true + eps))**2))

print("\n--- Results ---")
print(f"Final loss: {total_loss:.6f}")
print(f"Final MSE loss: {final_loss:.6f}")
print(f"RMSE:          {rmse:.6f}")
print(f"rel RMSE:          {rel_rmse:.6f}")
print(f"R^2:           {r2:.6f}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm, datasets
from ipywidgets import interact, FloatLogSlider, Dropdown, fixed


data = np.genfromtxt("../data/vcut_results_merged_OLD.txt", skip_header=1, delimiter=",")
data=data[:,:]
X = data[data[:, 4] != 0]
X = X[:, :3]
nnz = len(X[:, 0])
Xzero = data[data[:, 4] == 0]
Xzero = Xzero[:, :3]
X = np.vstack((X, Xzero))
y = np.zeros((X.shape[0],))
y[:nnz] = 1   # flatten y for sklearn

def plot_svm(C, gamma, kernel):
    clf = svm.SVC(C=C, gamma=gamma, kernel=kernel)
    clf.fit(X, y)

    # Count misclassifications
    y_pred = clf.predict(X)
    misclassified = np.sum(y_pred != y)
    print(f"Misclassified points: {misclassified} / {len(y)}")
    print(f"Accuracy: {100*(1-misclassified/len(y)):.3f}%")

    views = [(30, 30), (60, 30), (90, 30), (120, 30), (30, 0), (90, 0), (150, 0), (210, 0)]

    fig = plt.figure(figsize=(12, 6))
    for i, (az, el) in enumerate(views, 1):
        ax = fig.add_subplot(2, 4, i, projection='3d')
        ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=y, cmap=plt.cm.coolwarm, s=30, edgecolors='k')
        sv = clf.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], sv[:, 2], s=80, facecolors='none', edgecolors='k', linewidths=1.5)
        ax.view_init(elev=el, azim=az)
        ax.set_title(f"{kernel} kernel\naz={az}°, el={el}°")
        ax.set_xlabel('alpha')
        ax.set_ylabel('gamma')
        ax.set_zlabel('phi')

    plt.tight_layout()
    plt.show()
    return clf

clf = plot_svm(C=20, gamma=0.5, kernel='rbf')


In [ ]:
import joblib
import torch
import numpy as np

# SAVE SVM
joblib.dump(clf, "../model/svm_model.pkl")

# SAVE NN
torch.save(model.state_dict(), "../model/nn_model.pth")

# SAVE NORMALIZATION PARAMETERS
np.savez("../model/normalization.npz",
         X_mu=X_mu.numpy(),
         X_sigma=X_sigma.numpy(),
         Y_mu=Y_mu.numpy(),
         Y_sigma=Y_sigma.numpy())
